In [1]:
# 导入所有必需的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops

# 05. LLaMA3 Block Tutorial | 经典模型搭建: LLaMA-3 Transformer Block

> **LLaMA 架构 vs 传统 Transformer (如 GPT-2)**
> 1. **归一化位置 (Pre-Norm vs Post-Norm)**：LLaMA 使用 Pre-Norm（在 Attention 和 MLP **之前**进行归一化），这让深层网络的训练更加稳定；而早期模型多用 Post-Norm。
> 2. **归一化算法**：将 LayerNorm 替换为无偏置、不减均值的 **RMSNorm**，提升计算效率。
> 3. **激活函数**：将 ReLU/GELU 替换为 **SwiGLU**，通过门控机制（Gating）显著提升了模型的表达能力。
> 4. **位置编码**：彻底抛弃绝对位置编码，拥抱 **RoPE**。
> 5. **注意力机制**：从 LLaMA-2 开始，为了优化推理时的 KV Cache，将标准 MHA 升级为 **GQA**。

### 模块集成框架
LLaMA-3 前向传播的具体流程为：
1. 输入经过 Attention 层的 RMSNorm。
   $$ \text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2 + \epsilon} $$
   $$ y = \frac{x}{\text{RMS}(x)} \odot \gamma $$

2. 执行带 KV Cache 的 GQA 注意力机制（内含 RoPE 旋转）。
$$ \text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

3. 将残差相加：`x = x + attn_out`。
$$ h = x + \text{Attention}(\text{RMSNorm}(x)) $$

5. 经过 MLP 层的 RMSNorm。
   $$ \text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2 + \epsilon} $$
   $$ y = \frac{x}{\text{RMS}(x)} \odot \gamma $$

6. 执行 SwiGLU 前馈网络。
$$ \text{SwiGLU}(x) = (\text{Swish}(x W_{\text{gate}}) \otimes (x W_{\text{up}})) W_{\text{down}} $$
其中 $\text{Swish}(z) = z \cdot \sigma(z)$ (在 PyTorch 中对应 `F.silu`)。注意，为了保持参数量与传统 MLP 一致，LLaMA 中的隐藏层维度通常设置为 $\frac{8}{3} d$ 并向上取整。

7. 再次加上残差.
$$ \text{out} = h + \text{MLP}(\text{RMSNorm}(h)) $$




In [2]:
# ---------------------------------------------------------
# 以下是我们之前实现的组件 (此处用极简占位符代替，以保持代码整洁)
# ---------------------------------------------------------
class DummyRMSNorm(nn.Module):
    def __init__(self, dim): super().__init__(); self.w = nn.Parameter(torch.ones(dim))
    def forward(self, x): return x * self.w

class DummyAttention(nn.Module):
    def __init__(self, dim): super().__init__(); self.proj = nn.Linear(dim, dim)
    def forward(self, x): return self.proj(x) # 假装它做了 RoPE 和 GQA
# ---------------------------------------------------------

class LlamaMLP(nn.Module):
    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        # ==========================================
        # TODO 1: 定义 SwiGLU 所需的三个线性层 (无 bias)
        # ==========================================
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 2: 实现 SwiGLU 的前向传播
        # ==========================================
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))
        pass

class LlamaDecoderLayer(nn.Module):
    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        
        # 1. 注意力模块与它的前置 LayerNorm
        self.input_layernorm = DummyRMSNorm(hidden_size)
        self.self_attn = DummyAttention(hidden_size)
        
        # 2. MLP 模块与它的前置 LayerNorm
        self.post_attention_layernorm = DummyRMSNorm(hidden_size)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 3: 实现 LLaMA 的 Pre-Norm 残差连接
        # ==========================================
        
       # Attention Block
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states)
        hidden_states = residual + hidden_states

        # MLP Block
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states

        return hidden_states
        pass


**进阶思考：LLaMA 架构的五大创新**

1. **Pre-Norm 拓扑：** 相比 Post-Norm，训练更稳定，支持更深的网络。
2. **RMSNorm 替代 LayerNorm：** 去除均值计算和偏置，速度提升约 10-15%，且在大规模训练中表现相当。
3. **SwiGLU 激活函数：** 门控机制带来更强的表达能力，在多个基准测试中优于 GELU 和 ReLU。
4. **RoPE 位置编码：** 相对位置编码，支持长度外推，是当前大模型的标配。
5. **GQA 注意力：** 在 LLaMA-2/3 中引入，大幅减少 KV Cache 显存占用（相比 MHA 减少 8 倍），同时保持接近 MHA 的性能。

**工程实践：**
- **LLaMA-3 8B**：32 层 Decoder Layer，hidden_size=4096，32 个 Query 头，8 个 KV 头（GQA 比例 4:1）。
- **LLaMA-3 70B**：80 层 Decoder Layer，hidden_size=8192，64 个 Query 头，8 个 KV 头（GQA 比例 8:1）。
- **训练技巧：** 使用 BF16 混合精度训练，梯度裁剪（clip_grad_norm=1.0），AdamW 优化器（β1=0.9, β2=0.95），余弦学习率衰减。
